# Визуализация Hi-C матриц 

В этом блокноте мы загрузим `.mcool` файл и отрисуем карту контактов для 2 хромосомы

In [ ]:
import cooler
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cooltools
import seaborn as sns
plt.rcParams['figure.figsize'] = (8, 8)
from pathlib import Path


## 1. Загрузка данных
Подгружаем `.mcool` файл. Если этот блокнот лежит в папке `notebooks/`, а данные в `data/`, то путь будет начинаться с `../`

In [ ]:
resolution = 1_000_000
mcool_path = Path('../data/MoPh7_enr_v2.mcool')
clr = cooler.Cooler(f'{mcool_path}::/resolutions/{resolution}')

print("Chromosomes available:", clr.chromnames[:5])


Самая простая отрисовка целой хромосомы

In [ ]:
chrom = clr.chromnames[0]
matrix = clr.matrix(balance=False).fetch(chrom)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.matshow(matrix, norm=LogNorm(vmin=1, vmax=1000))
plt.colorbar(im, label='Raw Contact Frequency', fraction=0.046, pad=0.04)
ax.set_title(f'Full Chromosome: {chrom} (1 Mb resolution)', y=1.05)
plt.show()

Чтобы зазумиться в интересующее место, нужно задать его коонринаты и по ним вытащить матрицу (можно еще задать нужное разрешение)

In [ ]:
res = 100_000
clr_100kb = cooler.Cooler(f'{mcool_path}::/resolutions/{res}')

start, end = 10_000_000, 30_000_000
region = f'{chrom}:{start}-{end}'

matrix_zoom = clr_100kb.matrix(balance=False).fetch(region)

fig, ax = plt.subplots(figsize=(8, 8))
# We could use a different colormap ('Reds') here for variety (another colormap "fall")
im = ax.matshow(matrix_zoom, norm=LogNorm(vmin=1, vmax=200), cmap='Reds')
plt.colorbar(im, label='Raw Contact Frequency', fraction=0.046, pad=0.04)
ax.set_title(f'Zoomed Region: {region} (100 kb resolution)', y=1.05)

Посмотрим на 2 и 17 хромосомы

In [ ]:
chrom1 = '2'
chrom2 = '17'

# 1. Вытаскиваем нужные куски
mat1_1 = clr.matrix(balance=False).fetch(chrom1, chrom1)
mat1_2 = clr.matrix(balance=False).fetch(chrom1, chrom2)
mat2_1 = clr.matrix(balance=False).fetch(chrom2, chrom1)
mat2_2 = clr.matrix(balance=False).fetch(chrom2, chrom2)

# 2. Склеиваем их в одну матрицу
top_half = np.hstack([mat1_1, mat1_2])
bottom_half = np.hstack([mat2_1, mat2_2])
combined_mat = np.vstack([top_half, bottom_half])

# 3. Считаем позиции для подписей осей
len1 = mat1_1.shape[0]
len2 = mat2_2.shape[0]
tick_positions = [len1 / 2, len1 + len2 / 2]

f, ax = plt.subplots(figsize=(7, 6))
im = ax.matshow(combined_mat, vmax=500, cmap='Reds')
plt.colorbar(im, fraction=0.046, pad=0.04, label='raw counts')

# Настраиваем оси
ax.set(xticks=tick_positions, xticklabels=[chrom1, chrom2],
       yticks=tick_positions, yticklabels=[chrom1, chrom2],
       xlabel='position, chrom#', ylabel='position, chrom#')
ax.xaxis.set_label_position('top')

# Опционально: добавим тонкие линии, чтобы визуально разделить хромосомы
ax.axhline(len1, color='black', linewidth=0.5, linestyle='--')
ax.axvline(len1, color='black', linewidth=0.5, linestyle='--')

plt.show()